# 12 · Developer workflow: where things live, which tests guard them

**Goal:** before changing anything, know which module owns the behaviour, which tests pin it,
and which rules the repo enforces. Notebook 13 applies this to concrete changes.

In [ ]:
from nbkit import guardrails, pytest

## 1 · Code map

| Package | Folder | Owns | Talks to |
| --- | --- | --- | --- |
| `dtrack_left` (on-prem, CLI `dtrack`) | `cli/` | click commands, shared options, config loading | everything below |
| | `checks/` | Stage 1 left counts, Stage 2 pull + stats + commit | backends, common |
| | `backends/` | Left sources (`sources.py`), S3 store (`store.py`), Databricks helper (`databricks.py`, `dtrack tools dbx` only) | databases, files, S3 |
| | `common/` | dates, hashing, stats, manifest: the protocol rules | nothing |
| `dtrack_right` (Databricks job) | `job.py`, `notebook.py` | the job lifecycle and its notebook entry | checks, report |
| | `checks/` | reconcile + gate + manifest; drain + compare | backends, common |
| | `backends/` | `SparkTarget`, S3 and Volume stores | Spark, S3, Volumes |
| | `report/` | summary, CSVs, HTML, Excel, publishing | the work store |
| | `common/` | Spark-only protocol subset: audit, dates, hashing, manifest, sampling | nothing |

Tests mirror the source tree: `src/dtrack_left/backends/store.py` → `tests/unit/dtrack_left/backends/test_store.py`.
Shared fakes and fixtures live in `tests/support.py` (`MemoryStore`, `FakeConnection`, `FakeSpark`,
`synthetic_data`, `pairs_config`, `assert_store_contract`, `oracle_container`).

## 2 · Test tiers and markers

| Tier | Folder | Marker (flag) | Docker | Cloud | Speed | Use it to |
| --- | --- | --- | --- | --- | --- | --- |
| unit | `tests/unit` | none, always runs | no | no | ~20 s, ~400 tests, 99% coverage | pin every function and branch |
| local | `tests/local/s3` | `moto` (`--run-moto`) | yes | no | seconds | S3 wire behaviour against a real S3 API |
| local | `tests/local/spark` | `spark` (`--run-spark`) | no (Java) | no | ~15 s | the generated Spark SQL really runs |
| local | `tests/local/oracle` | `oracle` (`--run-oracle`) | yes | no | ~45 s | real Oracle SQL and types |
| local | `tests/local/e2e` | `e2e` (`--run-e2e`) | yes | no | ~1 min | both processes, whole protocol |
| live | `tests/live` | `live` (`--run-live`) | 2 of 5 files | yes | 1–3 min | Databricks, Volumes, a real job |

`docker` is orthogonal: `-m "not docker"` drops container tests on a machine without Docker.

In [ ]:
pytest("tests/unit")

## 3 · Rules the repo enforces

Some rules are tests (parity, packaging boundary, coverage); some are conventions checked here.
Run this cell after any structural change.

In [ ]:
guardrails()

The one docstring is the pre-existing module docstring of `dtrack_left/__init__.py`; the rule
is to strip such things only when that code is rewritten anyway. `notebook.py` is exempt because
`# MAGIC` lines are the Databricks notebook source format.

Rules enforced by tests:

| Rule | Test |
| --- | --- |
| shared normalisation and Spark sampling behavior agree | `tests/unit/common/test_parity.py` |
| Left base dependencies hold no Databricks package; the SDK only in `dtrack[dbx]`, lazily | `tests/unit/test_packaging.py` |
| Right contains only its allowed Spark runtime modules | `tests/unit/test_packaging.py` |
| every store honours one contract | `tests/support.py::assert_store_contract` |
| Spark SQL twins of Python rules agree | `test_spark_collapse_decimal_sql_applies_the_same_rule_as_python` |

## 4 · Change → tests

| You changed | Run at least | Also |
| --- | --- | --- |
| a portable protocol rule | update only the required Left/Right modules; `pytest tests/unit/common` | behavior parity; `just e2e`; a Spark twin if the rule has one (`targets.py`) |
| a Left source (`backends/sources.py`) | `tests/unit/dtrack_left/backends/test_sources.py` | `just test oracle` for Oracle SQL; live step 3 |
| a store (`backends/store.py`, either side) | both `test_store.py` | `just test moto`; live `dtrack_left/test_s3.py`, `test_volume.py` |
| Spark SQL (`backends/targets.py`) | `tests/unit/dtrack_right/backends/test_targets.py` | `just test spark`; live `test_databricks_target.py` |
| gate / reconcile / manifest | both `checks/test_row_check.py`, `common/test_manifest.py`, `test_protocol_flow.py` | `just e2e` |
| Stage 2 pipeline or scheduler | both `checks/test_col_check.py`, `test_protocol_flow.py` | `just e2e`; live `test_job_flow.py` |
| report | `tests/unit/dtrack_right/report/` | open `examples/e2e/output/report/index.html` after `just e2e` |
| a CLI option or command | `tests/unit/dtrack_left/cli/` | `dtrack <cmd> -h`; notebook 07 |
| the protocol (file names, fields, unit ids) | everything above | bump `PROTOCOL_VERSION`; `just e2e`; live `test_job_flow.py` |

## 5 · Everyday commands

```bash
just test unit                       # the whole unit tier
uv run pytest tests/unit -k gate     # one topic
uv run pytest tests/unit --lf        # only what failed last time
just test cov                        # coverage with missing lines
just test local                      # moto + spark + oracle
just e2e                             # both processes on Moto
just live test -m "not docker"       # cloud only
DTRACK_LIVE_KEEP=1 just live test …  # keep what the live test created, to inspect it
just test debug -- tests/unit/...    # stop in ipdb
just dev lint && just dev fmt        # ruff on src and tests
```

## 6 · Definition of done

1. A unit test that failed before the change and passes after it.
2. `just test unit` green, coverage not lower (`just test cov`).
3. The tier that owns the real dependency, from the table above.
4. `just dev lint` clean; `guardrails()` all True except the known exceptions.
5. Docs: the matching `docs/notes/*.md`, `.env.example` for new variables, an ADR addendum when a
   boundary moves.
6. One lowercase commit line per logical change.